# Part 6 · Notebook 02 — Option chains and strike selection

**Sessions:** S2 (Option chain & strike library) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Build OCC option symbols.
2. Filter a chain to quotes you could actually trade.
3. Pick the at-the-money strike on the **forward**, not on spot.
4. Find the strikes one expected move away.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()
from datetime import date

## 1. A chain

One expiry, 30 days out, spot 600, rate 4.5%, dividend yield 1.3%. Prices come from a known smile (`iv_true`) with bid/ask spreads that widen in the wings, zero bids far out, and open interest concentrated near the money.

In [ ]:
S, T, r, q = 600.0, 30 / 365, 0.045, 0.013
chain = p.synthetic_chain(S, T, r, q)
calls, puts = chain[chain.cp == 1], chain[chain.cp == -1]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(calls.strike, calls.mid, label="calls"); axes[0].plot(puts.strike, puts.mid, label="puts"); axes[0].set_title("Mid prices"); axes[0].legend()
axes[1].bar(calls.strike, calls.oi, width=4); axes[1].set_title("Open interest (calls)")
plt.tight_layout(); plt.show()
chain.head()

## 2. OCC symbols

Every listed US option has a 21-character-style OCC symbol: **root + YYMMDD + C/P + strike × 1000 as 8 digits**, e.g. `AAPL261218C00200000` is the AAPL 18 Dec 2026 200 call. (Brokers may pad the root to six characters; we don't.)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def occ_symbol(root, expiry, cp, strike):
    return ...                                    # ✍️ f-string: {expiry:%y%m%d} and {int(round(strike * 1000)):08d} help

cases = [("AAPL", date(2026, 12, 18), "C", 200.0), ("SPY", date(2025, 3, 21), "P", 572.5), ("QQQ", date(2025, 1, 17), "C", 0.5)]
mine = [occ_symbol(*c) for c in cases]
mine = p.check("occ_symbol", mine, [p.occ_symbol(*c) for c in cases])
[(m, p.parse_occ(m)) for m in mine]

## 3. Quotes you can trade

Keep a row only if the **bid is above zero**, the ask is at least the bid, the relative spread `(ask − bid)/mid` is at most `max_spread_pct`, and open interest is at least `min_oi`. Everything downstream (implied vols, the smile, strike selection) should use the filtered chain.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def liquidity_filter(chain, max_spread_pct=0.10, min_oi=100):
    spread = (chain["ask"] - chain["bid"]) / chain["mid"]
    keep = ...                                    # ✍️ the four conditions, combined with &
    return chain[keep]

mine = p.attempt(liquidity_filter, chain)
mine = p.check("liquidity_filter", mine, p.liquidity_filter(chain))
print(f"kept {len(mine)} of {len(chain)} quotes; strikes kept: {mine.strike.min():.0f}–{mine.strike.max():.0f}")

## 4. At the money means at the forward

With a positive carry (r > q) the forward is above spot, and the strike where calls and puts are worth the same is near the **forward**. Choose the listed strike nearest to it (ties go to the lower strike).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def atm_strike(strikes, forward):
    k = np.sort(np.asarray(strikes, dtype=float))
    return ...                                    # ✍️ the strike with the smallest |k − forward| (argmin picks the first, i.e. lower, on a tie)

F = p.fair_value(S, r, q, T)
strikes = calls.strike.to_numpy()
cases = [(strikes, F), (strikes, S), (strikes, 602.5), (np.arange(90, 111, 1.0), 100.5)]
mine = [atm_strike(k, f) for k, f in cases]
mine = p.check("atm_strike", mine, [p.atm_strike(k, f) for k, f in cases])
print(f"30 days: forward {F:.2f} → ATM strike {mine[0]:.0f}; spot {S:.0f} → {mine[1]:.0f} (the same here)")
F1y = p.fair_value(S, r, q, 1.0)
print(f"1 year:  forward {F1y:.2f} → ATM strike {p.atm_strike(np.arange(400, 801, 5.0), F1y):.0f}; spot → 600: a whole 20 points off")

In [ ]:
c_at = calls.set_index("strike").model_mid; p_at = puts.set_index("strike").model_mid
for k in (600.0, 605.0):
    print(f"K = {k:.0f}: call − put = {c_at[k] - p_at[k]:+.3f}")
print("the strike where call − put changes sign is the forward: that's where 'at the money' is")

## 5. One expected move

A common strike rule: sell the strikes one **expected move** away, `S·(1 ± σ√T)` with the ATM implied vol.

In [ ]:
atm_iv = float(calls.set_index("strike").iv_true[p.atm_strike(strikes, F)])
lo, hi = p.expected_move_strikes(strikes, S, atm_iv, T)
print(f"ATM IV {atm_iv:.1%} → expected move ±{S * atm_iv * np.sqrt(T):.1f} → strikes {lo:.0f} / {hi:.0f}")

## Wrap-up

* OCC symbols identify contracts across brokers.
* Filter before you compute anything; stale and one-sided quotes make holes in the smile.
* ATM and moneyness are measured against the forward.
* Graded version: `labs/part06/week21_futures_chains` (IB and Alpaca chains into one schema, expiry selection, strikes by delta).